# Forecasting Time Series Bulanan (12 Bulan per Tahun)
Notebook ini membandingkan beberapa model bulanan, memilih model terbaik berdasarkan metrik validasi, lalu membuat prediksi 12 bulan ke depan.

## Alur
1. Load data gempa
2. Bentuk deret waktu bulanan
3. Split train/test (12 bulan terakhir sebagai test)
4. Latih dan evaluasi model kandidat (Seasonal Naive, Holt-Winters, SARIMA, XGBoost, LSTM)
5. Pilih model terbaik dan forecast 12 bulan ke depan
6. Simpan artifact (metrics, prediksi test, train/test split, dan forecast)

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Optional: XGBoost
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False

# Optional: TensorFlow/Keras untuk LSTM
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense
    from tensorflow.keras.callbacks import EarlyStopping
    TF_AVAILABLE = True
except ImportError:
    TF_AVAILABLE = False

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

if TF_AVAILABLE:
    tf.random.set_seed(42)

print(f'XGBoost available: {XGB_AVAILABLE}')
print(f'TensorFlow available: {TF_AVAILABLE}')

In [ ]:
# Konfigurasi utama
DATA_CANDIDATES = [
    Path('data_gempa_kaggle/katalog_gempa.csv'),
    Path('../data_gempa_kaggle/katalog_gempa.csv'),
    Path('timeseries_triwulan/data_gempa_kaggle/katalog_gempa.csv')
]
DATE_CANDIDATES = ['time', 'datetime', 'date', 'tanggal', 'origin_time']
TARGET_MODE = 'count'  # opsi: 'count' atau nama kolom numerik (mis. 'magnitude')
TEST_HORIZON = 12
SEASONAL_PERIOD = 12

def find_existing_path(candidates):
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError('File data tidak ditemukan. Sesuaikan DATA_CANDIDATES.')

data_path = find_existing_path(DATA_CANDIDATES)
df_raw = pd.read_csv(data_path)
print(f'Data loaded: {data_path}')
print(f'Shape: {df_raw.shape}')
df_raw.head()

In [ ]:
def detect_date_column(df, date_candidates):
    lowered = {c.lower(): c for c in df.columns}
    for cand in date_candidates:
        if cand in lowered:
            return lowered[cand]

    best_col = None
    best_valid = -1
    for col in df.columns:
        parsed = pd.to_datetime(df[col], errors='coerce')
        valid = parsed.notna().sum()
        if valid > best_valid:
            best_valid = valid
            best_col = col

    if best_col is None or best_valid == 0:
        raise ValueError('Tidak ada kolom tanggal yang bisa diparse.')
    return best_col

date_col = detect_date_column(df_raw, DATE_CANDIDATES)
df = df_raw.copy()
df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
df = df.dropna(subset=[date_col]).sort_values(date_col)

if TARGET_MODE == 'count':
    ts = df.set_index(date_col).resample('MS').size().astype(float)
    target_name = 'monthly_event_count'
else:
    if TARGET_MODE not in df.columns:
        raise ValueError(f'Kolom target {TARGET_MODE} tidak ditemukan.')
    ts = (
        df.set_index(date_col)[TARGET_MODE]
        .astype(float)
        .resample('MS')
        .mean()
    )
    target_name = TARGET_MODE

ts = ts.asfreq('MS').interpolate(method='linear')

if len(ts) < (TEST_HORIZON + 24):
    raise ValueError('Data terlalu pendek. Butuh minimal sekitar 36 observasi bulanan.')

print('Kolom tanggal terdeteksi:', date_col)
print('Target:', target_name)
print('Rentang data:', ts.index.min().date(), 'sampai', ts.index.max().date())
print('Jumlah titik bulanan:', len(ts))
ts.head()

In [ ]:
train = ts.iloc[:-TEST_HORIZON]
test = ts.iloc[-TEST_HORIZON:]

N_LAGS = 12

def mape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def metric_frame(name, y_true, y_pred):
    return {
        'model': name,
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAPE': mape(y_true, y_pred)
    }

def make_supervised(values, n_lags):
    X, y = [], []
    for i in range(n_lags, len(values)):
        X.append(values[i - n_lags:i])
        y.append(values[i])
    return np.array(X), np.array(y)

def recursive_forecast_sklearn(model, history_values, steps, n_lags):
    history = list(history_values)
    preds = []
    for _ in range(steps):
        x_input = np.array(history[-n_lags:]).reshape(1, -1)
        pred = float(model.predict(x_input)[0])
        preds.append(pred)
        history.append(pred)
    return np.array(preds)

def recursive_forecast_lstm(model, history_values, steps, n_lags):
    history = list(history_values)
    preds = []
    for _ in range(steps):
        x_input = np.array(history[-n_lags:]).reshape(1, n_lags, 1)
        pred = float(model.predict(x_input, verbose=0).ravel()[0])
        preds.append(pred)
        history.append(pred)
    return np.array(preds)

def post_process_forecast(values):
    values = np.array(values, dtype=float)
    if TARGET_MODE == 'count':
        values = np.clip(values, 0, None)
    return values

results = []
predictions = {}

In [ ]:
# Model 1: Seasonal Naive
last_year = train.iloc[-SEASONAL_PERIOD:].values
pred_naive = np.resize(last_year, TEST_HORIZON)
pred_naive = post_process_forecast(pred_naive)
pred_naive = pd.Series(pred_naive, index=test.index)
predictions['SeasonalNaive'] = pred_naive
results.append(metric_frame('SeasonalNaive', test, pred_naive))

# Model 2: Holt-Winters Exponential Smoothing
hw = ExponentialSmoothing(
    train,
    trend='add',
    seasonal='add',
    seasonal_periods=SEASONAL_PERIOD
).fit(optimized=True)
pred_hw = post_process_forecast(hw.forecast(TEST_HORIZON).values)
pred_hw = pd.Series(pred_hw, index=test.index)
predictions['HoltWintersAdd'] = pred_hw
results.append(metric_frame('HoltWintersAdd', test, pred_hw))

# Model 3: SARIMA grid kecil, pilih berdasarkan AIC di train
best_aic = np.inf
best_order = None
best_seasonal = None
best_sarima_fit = None

for order in [(1,0,1), (1,1,1), (2,1,1), (2,1,2)]:
    for seas in [(0,1,1,SEASONAL_PERIOD), (1,1,1,SEASONAL_PERIOD), (1,0,1,SEASONAL_PERIOD)]:
        try:
            fit = SARIMAX(
                train,
                order=order,
                seasonal_order=seas,
                enforce_stationarity=False,
                enforce_invertibility=False
            ).fit(disp=False)
            if fit.aic < best_aic:
                best_aic = fit.aic
                best_order = order
                best_seasonal = seas
                best_sarima_fit = fit
        except Exception:
            continue

if best_sarima_fit is not None:
    pred_sarima = post_process_forecast(best_sarima_fit.forecast(TEST_HORIZON).values)
    pred_sarima = pd.Series(pred_sarima, index=test.index)
    predictions['SARIMA'] = pred_sarima
    results.append(metric_frame('SARIMA', test, pred_sarima))
    print('Best SARIMA:', best_order, best_seasonal, 'AIC=', round(best_aic, 2))
else:
    print('SARIMA gagal dilatih pada kombinasi yang dicoba.')

# Model 4: XGBoost Regressor (lag features)
xgb_model = None
if XGB_AVAILABLE and len(train) > N_LAGS:
    X_train_xgb, y_train_xgb = make_supervised(train.values, N_LAGS)
    xgb_model = xgb.XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective='reg:squarederror',
        random_state=42
    )
    xgb_model.fit(X_train_xgb, y_train_xgb)
    pred_xgb = recursive_forecast_sklearn(xgb_model, train.values, TEST_HORIZON, N_LAGS)
    pred_xgb = post_process_forecast(pred_xgb)
    pred_xgb = pd.Series(pred_xgb, index=test.index)
    predictions['XGBoost'] = pred_xgb
    results.append(metric_frame('XGBoost', test, pred_xgb))
else:
    print('XGBoost dilewati (package tidak tersedia atau data train terlalu pendek).')

# Model 5: LSTM (lag sequence)
lstm_model = None
lstm_scaler = None
if TF_AVAILABLE and len(train) > N_LAGS + 6:
    lstm_scaler = MinMaxScaler()
    train_scaled = lstm_scaler.fit_transform(train.values.reshape(-1, 1)).ravel()
    X_train_lstm, y_train_lstm = make_supervised(train_scaled, N_LAGS)
    X_train_lstm = X_train_lstm.reshape(X_train_lstm.shape[0], X_train_lstm.shape[1], 1)

    lstm_model = Sequential([
        LSTM(32, input_shape=(N_LAGS, 1)),
        Dense(1)
    ])
    lstm_model.compile(optimizer='adam', loss='mse')

    es = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
    lstm_model.fit(
        X_train_lstm,
        y_train_lstm,
        epochs=120,
        batch_size=8,
        verbose=0,
        callbacks=[es]
    )

    pred_lstm_scaled = recursive_forecast_lstm(
        lstm_model, train_scaled, TEST_HORIZON, N_LAGS
    )
    pred_lstm = lstm_scaler.inverse_transform(pred_lstm_scaled.reshape(-1, 1)).ravel()
    pred_lstm = post_process_forecast(pred_lstm)
    pred_lstm = pd.Series(pred_lstm, index=test.index)
    predictions['LSTM'] = pred_lstm
    results.append(metric_frame('LSTM', test, pred_lstm))
else:
    print('LSTM dilewati (TensorFlow tidak tersedia atau data train terlalu pendek).')

In [ ]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
best_model_name = results_df.loc[0, 'model']

display(results_df)
print(f'Model terbaik (berdasarkan MAE): {best_model_name}')

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(train.index, train.values, label='Train')

# Jembatani garis train -> test supaya visual nyambung
test_bridge_idx = pd.DatetimeIndex([train.index[-1], *test.index])
test_bridge_val = np.concatenate([[train.iloc[-1]], test.values])
plt.plot(test_bridge_idx, test_bridge_val, label='Test', linewidth=2)

for name, pred in predictions.items():
    pred_bridge_idx = pd.DatetimeIndex([train.index[-1], *pred.index])
    pred_bridge_val = np.concatenate([[train.iloc[-1]], pred.values])
    plt.plot(pred_bridge_idx, pred_bridge_val, label=f'Pred-{name}', linestyle='--')

plt.title('Perbandingan Prediksi pada Data Test (12 Bulan)')
plt.legend()
plt.show()

In [ ]:
# Retrain model terbaik dengan seluruh data, lalu forecast 12 bulan ke depan
full = ts.copy()
future_idx = pd.date_range(full.index[-1] + pd.offsets.MonthBegin(1), periods=12, freq='MS')

if best_model_name == 'SeasonalNaive':
    vals = full.iloc[-SEASONAL_PERIOD:].values
    future_fc = np.resize(vals, 12)

elif best_model_name == 'HoltWintersAdd':
    fit_best = ExponentialSmoothing(
        full, trend='add', seasonal='add', seasonal_periods=SEASONAL_PERIOD
    ).fit(optimized=True)
    future_fc = fit_best.forecast(12).values

elif best_model_name == 'SARIMA':
    fit_best = SARIMAX(
        full,
        order=best_order,
        seasonal_order=best_seasonal,
        enforce_stationarity=False,
        enforce_invertibility=False
    ).fit(disp=False)
    future_fc = fit_best.forecast(12).values

elif best_model_name == 'XGBoost':
    if not XGB_AVAILABLE:
        raise RuntimeError('XGBoost tidak tersedia pada environment ini.')
    X_full_xgb, y_full_xgb = make_supervised(full.values, N_LAGS)
    fit_best = xgb.XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective='reg:squarederror',
        random_state=42
    )
    fit_best.fit(X_full_xgb, y_full_xgb)
    future_fc = recursive_forecast_sklearn(fit_best, full.values, 12, N_LAGS)

elif best_model_name == 'LSTM':
    if not TF_AVAILABLE:
        raise RuntimeError('TensorFlow tidak tersedia pada environment ini.')
    scaler_full = MinMaxScaler()
    full_scaled = scaler_full.fit_transform(full.values.reshape(-1, 1)).ravel()
    X_full_lstm, y_full_lstm = make_supervised(full_scaled, N_LAGS)
    X_full_lstm = X_full_lstm.reshape(X_full_lstm.shape[0], X_full_lstm.shape[1], 1)

    fit_best = Sequential([
        LSTM(32, input_shape=(N_LAGS, 1)),
        Dense(1)
    ])
    fit_best.compile(optimizer='adam', loss='mse')
    es = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
    fit_best.fit(
        X_full_lstm,
        y_full_lstm,
        epochs=120,
        batch_size=8,
        verbose=0,
        callbacks=[es]
    )

    future_scaled = recursive_forecast_lstm(fit_best, full_scaled, 12, N_LAGS)
    future_fc = scaler_full.inverse_transform(future_scaled.reshape(-1, 1)).ravel()

else:
    raise ValueError('Nama model tidak dikenali.')

future_fc = post_process_forecast(future_fc)
forecast_12m = pd.DataFrame({
    'month': future_idx,
    'forecast': future_fc
})

display(forecast_12m)

plt.figure(figsize=(12, 5))
plt.plot(full.index, full.values, label='Historis')

# Jembatani garis historis -> forecast supaya visual nyambung
forecast_bridge_idx = pd.DatetimeIndex([full.index[-1], *future_idx])
forecast_bridge_val = np.concatenate([[full.iloc[-1]], future_fc])
plt.plot(forecast_bridge_idx, forecast_bridge_val, label='Forecast 12 Bulan', linestyle='--', linewidth=2)

plt.title(f'Forecast 12 Bulan ke Depan - {best_model_name}')
plt.legend()
plt.show()

In [ ]:
# Simpan artefak hasil
out_dir = Path('artifacts/model_monthly_auto_select')
out_dir.mkdir(parents=True, exist_ok=True)

# Simpan split train/test seperti artifact notebook lain
train_split_df = pd.DataFrame({
    'month': train.index,
    'target': train.values
})
test_split_df = pd.DataFrame({
    'month': test.index,
    'target': test.values
})

# Simpan prediksi test tiap model
pred_test_df = pd.DataFrame({
    'month': test.index,
    'actual': test.values
})
for model_name, pred in predictions.items():
    pred_test_df[f'pred_{model_name}'] = pred.values

results_df.to_csv(out_dir / 'metrics_comparison.csv', index=False)
pred_test_df.to_csv(out_dir / 'predictions_test.csv', index=False)
train_split_df.to_csv(out_dir / 'train_split.csv', index=False)
test_split_df.to_csv(out_dir / 'test_split.csv', index=False)
forecast_12m.to_csv(out_dir / 'forecast_next_12_months.csv', index=False)

meta = {
    'data_path': str(data_path),
    'date_column': date_col,
    'target': target_name,
    'test_horizon': TEST_HORIZON,
    'seasonal_period': SEASONAL_PERIOD,
    'n_lags': N_LAGS,
    'xgboost_available': XGB_AVAILABLE,
    'tensorflow_available': TF_AVAILABLE,
    'best_model': best_model_name
}
pd.Series(meta).to_json(out_dir / 'metadata.json', indent=2)

print('Artefak tersimpan di:', out_dir.resolve())